# CU-BEMS Intelligence Suite — Question 1 (Operational Baseload)
## Notebook 1E: Exploratory Data Analysis & Baseload Modeling (`qn1e_eda_and_baseload_modeling.ipynb`)

---

### Core Research Question
> *How can we identify and reduce unnecessary energy consumption outside operating hours across all 7 floors of the Chamchuri 5?*

**Data Scope:** 18 continuous months (July 1, 2018 – December 31, 2019) across 33 lighting zones and 32 plug circuits (65 total electrical sub-meters).

---

### Expected Project Deliverables
1. **01 Waste Map**: A per-circuit, per-floor view of where after-hours baseload runs and how large it is across all 7 floors.
2. **02 Circuit Classification**: Each of the 65 circuits tagged as **Critical Standby** or **Preventable Waste** via unsupervised machine learning.

---

### Methodological Architecture
1. **Operational Schedule Stratification**: Stratify the 18-month timeline into Occupied (Mon–Fri 08:00–17:00, 25.0% of calendar) vs. Unoccupied (Nights, Weekends, 36 statutory Thai National Holidays, 75.0% of calendar).
2. **Empirical Schedule Validation**: Validate the 08:00–17:00 operating window by comparing Weekday vs. Saturday hourly load profiles, demonstrating a sustained +85 kW daytime occupancy surge and evening load decline.
3. **Deep-Night Reference Calibration**: Establish reference baseload during the verified empirical load trough (Sundays 02:00–04:00 AM) across 52 training weeks.
4. **Dynamic Time-of-Week Quantile Formulation ($P_{2.5}(d, h)$)**: Derive dynamic circuit baseload floors using the 2.5th percentile across time-of-week slots ($q=0.025$). This accounts for empirical operational transitions (e.g. evening shoulder departure vs. deep sleep) without absorbing human negligence.
5. **Unsupervised Machine Learning (K-Means, $k=2$)**: Cluster all 65 circuits across three physical dimensions (base-load ratio, baseload magnitude, and off-hours CV) to tag each circuit as **Critical Standby** (e.g., IT servers, security) vs. **Preventable Waste** (curfew-controllable office loads).
6. **Per-Circuit Spatial Waste Map**: Map after-hours standby loads and preventable schedule surplus across all 7 floors and 65 circuits under the dynamic baseline.
7. **Addressable Energy & Utility Cost Quantification**: Price preventable surplus against official MEA Schedule 3.2 / 4.2 Time-of-Use rates (12-24 kV Tier) (Peak ฿4.3297, Off-Peak ฿2.5849) broken down by building scope, end-use category, and floor (Floors 1 to 7).
8. **Digital Twin Synchronization**: Export the dynamic baseload matrix and classification registry directly to the building intelligence webapp and digital twin platform.

---

### Core Modeling Assumptions & Boundary Conditions
1. **Zero-Power Metrology Integrity**: A reading of `0.000 kW` represents a genuine physically de-energized circuit ($I = 0\text{ A}$), not missing data. True telemetry loss registers as `NaN` and is preserved without artificial interpolation.
2. **Schedule & Telemetry Occupancy Inference**: With no turnstile or RFID badge-swipe telemetry available, occupancy is inferred from facility operating hours (Monday–Friday 08:00–17:00), 36 statutory Bank of Thailand public holidays, and physically verified via daytime plug-load step changes (+85 kW surge).
3. **Reference Calibration Window**: Sunday 02:00–04:00 AM represents the verified empirical weekly load trough (~79.0 kW), capturing steady-state overnight load including timer-driven exterior lighting.
4. **Dynamic Non-Parametric Estimator ($P_{2.5}$)**: The 2.5th percentile across time-of-week slots accounts for physical departure gradients while providing a robust annual buffer against transient dropouts.
5. **Operational Waste vs. Hardware Faults**: Flagged after-hours loads represent **potential operational waste** (functional equipment left energized in unoccupied spaces), not confirmed hardware faults or physical electrical defects.
6. **Conservative Tariff Valuation**: Savings are evaluated strictly on energy-only MEA Schedule 4.2 TOU rates, excluding peak demand surcharges and Ft fees, broken down directly across the 61 curfew-eligible circuits while protecting the 4 K-Means-identified critical standby circuits.


In [1]:
# ==============================================================================
# 1. ENVIRONMENT & DIRECTORY CONFIGURATION
# ==============================================================================
import os, glob, json, math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, time

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'sans-serif']
plt.rcParams['axes.edgecolor'] = '#CCCCCC'
plt.rcParams['axes.linewidth'] = 0.8

data_dir = os.path.abspath(os.path.join(os.getcwd(), '..', 'data')) if os.path.basename(os.getcwd()) == 'notebooks' else os.path.join(os.getcwd(), 'data')

# Ingest official Bank of Thailand public holiday dates from qn1a
holidays_path = os.path.join(data_dir, 'thai_holidays_18months.csv')
if os.path.exists(holidays_path):
    holiday_dates = set(pd.to_datetime(pd.read_csv(holidays_path)['Date']).dt.date)
else:
    holiday_dates = set()


## 2. Harmonized Telemetry Loading (Floors 1 to 7)
We load the 18-month telemetry for all 7 floors, guaranteeing all calendar flags (`Hour`, `DayOfWeek`, `Is_Holiday`, `Is_Occupied`, `Is_Train_Period`, `Split`) are uniformly present.


In [2]:
# ==============================================================================
# 2. LOAD HARMONIZED FLOOR DATASETS (FLOORS 1 TO 7)
# ==============================================================================
floor_dfs = {}
summary_rows = []

for f in range(1, 8):
    floor_key = f"Floor{f}"
    f_path = os.path.join(data_dir, f"clean_merged_{floor_key}.csv")
    if os.path.exists(f_path):
        df = pd.read_csv(f_path, parse_dates=['Date'], index_col='Date')
        # Rebuild calendar columns if this copy of the file lacks them (e.g. hourly exports of NB 1D).
        # Same rules as qn1d_schedule_merge.ipynb.
        if 'DayOfWeek' not in df.columns:
            df['DayOfWeek'] = df.index.dayofweek
        if 'Hour' not in df.columns:
            df['Hour'] = df.index.hour
        if 'Is_Holiday' not in df.columns:
            df['Is_Holiday'] = pd.Series(df.index.date, index=df.index).isin(holiday_dates)
        if 'Is_Occupied' not in df.columns:
            df['Is_Occupied'] = (df['DayOfWeek'] < 5) & (~df['Is_Holiday']) & (df['Hour'] >= 8) & (df['Hour'] < 17)
            df['Is_Unoccupied'] = ~df['Is_Occupied']
        if 'Split' not in df.columns:
            df['Split'] = np.where(df.index <= pd.Timestamp('2019-06-30 23:59:59'), 'Train', 'Test')
        if df.index.to_series().diff().median() > pd.Timedelta(minutes=1):
            print(f"[WARN] {floor_key}: file is not minute-level. Section 5-6 baselines will differ from the "
                  f"minute-level run. Re-run qn1b -> qn1c -> qn1d to regenerate it.")
        floor_dfs[floor_key] = df
        n_light = len([c for c in df.columns if '_Light' in c])
        n_plug = len([c for c in df.columns if '_Plug' in c])
        summary_rows.append({
            'Floor': floor_key,
            'Total_Rows': f"{len(df):,}",
            'Lighting_Circuits': n_light,
            'Plug_Circuits': n_plug,
            'Total_Circuits': n_light + n_plug
        })

ingest_summary_df = pd.DataFrame(summary_rows)
display(ingest_summary_df)


,Floor,Total_Rows,Lighting_Circuits,Plug_Circuits,Total_Circuits
0,Floor1,"790,560",4,3,7
1,Floor2,"790,560",4,4,8
2,Floor3,"790,560",5,5,10
3,Floor4,"790,560",5,5,10
4,Floor5,"790,560",5,5,10
5,Floor6,"689,128",5,5,10
6,Floor7,"790,560",5,5,10


## 3. Empirical Baseline Calibration: 168-Hour Weekly Heatmap
We aggregate all lighting and plug power building-wide across the 18 months to identify the global empirical load trough.


In [3]:
# ==============================================================================
# 3. 168-HOUR BUILDING LOAD HEATMAP
# ==============================================================================
bldg_load = pd.concat([df[[c for c in df.columns if ('_Light(kW)' in c) or ('_Plug(kW)' in c)]] 
                       for df in floor_dfs.values()], axis=1).sum(axis=1)

weekly_matrix = bldg_load.groupby([bldg_load.index.dayofweek, bldg_load.index.hour]).mean().unstack()

plt.figure(figsize=(15, 6))
sns.heatmap(weekly_matrix, cmap='YlOrRd', annot=True, fmt='.1f', cbar_kws={'label': 'Mean Active Power (kW)'})
plt.title("Chamchuri 5: 168-Hour Average Weekly Electrical Load Profile (Lights + Plugs)", fontsize=13, fontweight='bold')
plt.xlabel("Hour of Day (00:00 - 23:00)", fontweight='bold')
plt.ylabel("Day of Week (0=Monday, 6=Sunday)", fontweight='bold')
plt.yticks(ticks=[0.5, 1.5, 2.5, 3.5, 4.5, 5.5, 6.5], labels=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'], rotation=0)

# Highlight deep sleep reference window: Sunday (day 6), hours 2 and 3 (02:00-04:00)
from matplotlib.patches import Rectangle
plt.gca().add_patch(Rectangle((2, 6), 2, 1, fill=False, edgecolor='#2563EB', lw=3, ls='--', label='Deep Sleep Ref (Sun 02:00-04:00)'))
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

min_val = weekly_matrix.loc[6, 2:3].mean()


/var/folders/1g/x_wv4b8n0zs5kkhz0j1mg7v40000gn/T/ipykernel_3494/249114802.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


> **Empirical Baseline Calibration Finding**:
> Across the 168-hour weekly operational profile, **Sunday 02:00–04:00 AM** represents the verified **lowest-occupancy, most stable overnight load window (79.0 kW average)** across the 18-month telemetry (80.7 kW in the 12-month training period). This captures steady-state quiescent machine baseload and timer-controlled exterior illumination during the period of lowest human activity.


### Engineering Audit: Why Sunday 02:00–04:00 AM Baseload Window (vs. 06:00–08:00 AM)
A surface-level review of the Sunday heatmap shows that the absolute minimum aggregate building load occurs at **06:00–08:00 AM (~55 kW average, dipping to 51.9 kW at 07:00)** rather than **02:00–04:00 AM (79.0 kW)**.

From a Smart Building & Building Management Systems (BEMS) engineering perspective, we perform a circuit-level audit to understand the physical drivers of this difference and justify our choice of the reference baseload calibration window:


In [4]:
# ==============================================================================
# 3B. ENGINEERING AUDIT: SUNDAY 02:00-04:00 VS 06:00-08:00 CALIBRATION
# ==============================================================================
master_hourly = pd.read_csv(os.path.join(data_dir, 'qn1_hourly_master_for_excel.csv'), parse_dates=['Date'], index_col='Date')
elec_cols = [c for c in master_hourly.columns if ('_Light(kW)' in c) or ('_Plug(kW)' in c)]

sun_hourly = master_hourly[master_hourly.index.dayofweek == 6]
circ_2_4 = sun_hourly[(sun_hourly.index.hour >= 2) & (sun_hourly.index.hour < 4)][elec_cols].mean()
circ_6_8 = sun_hourly[(sun_hourly.index.hour >= 6) & (sun_hourly.index.hour < 8)][elec_cols].mean()
circ_drop = (circ_2_4 - circ_6_8).sort_values(ascending=False)

audit_comparison_df = pd.DataFrame({
    'Sunday 02:00-04:00 (kW)': circ_2_4[circ_drop.index[:6]],
    'Sunday 06:00-08:00 (kW)': circ_6_8[circ_drop.index[:6]],
    'Load Drop (kW)': circ_drop.head(6),
    'Likely Inferred Function (Consistent with Day-Night Schedule Pattern)': [
        'Exterior Facade / Floodlights (Consistent with Sunrise Timer Shutoff)',
        'Entrance Canopy / Atrium Fixtures (Consistent with Daylight Switching)',
        'Ground Lobby General Receptacles',
        'Documented SGRU Living Lab / IT Server Room (Salama et al. 2021 Table 1)',
        'Floor 1 Zone 3 Miscellaneous Plugs',
        'Ground Floor Zone 2 Corridor Fixtures'
    ]
})
print('=== CIRCUIT DELTA: SUNDAY 02:00-04:00 VS 06:00-08:00 ===')
display(audit_comparison_df.round(2))


=== CIRCUIT DELTA: SUNDAY 02:00-04:00 VS 06:00-08:00 ===


,Sunday 02:00-04:00 (kW),Sunday 06:00-08:00 (kW),Load Drop (kW),Likely Inferred Function (Consistent with Day-Night Schedule Pattern)
Floor1_z4_Light(kW),22.16,9.25,12.90,Exterior Facade / Floodlights (Consistent with...
Floor1_z1_Light(kW),10.93,0.27,10.66,Entrance Canopy / Atrium Fixtures (Consistent ...
Floor1_z1_Plug(kW),3.62,1.29,2.33,Ground Lobby General Receptacles
Floor1_z2_Plug(kW),14.84,14.68,0.15,Documented SGRU Living Lab / IT Server Room (S...
Floor1_z3_Plug(kW),1.05,0.96,0.09,Floor 1 Zone 3 Miscellaneous Plugs
Floor1_z2_Light(kW),9.55,9.47,0.08,Ground Floor Zone 2 Corridor Fixtures


> **Physical Engineering Rationale (Sunday 02:00–04:00 AM vs. 06:00–08:00 AM)**:
> * **Aggregate Dip at Sunrise**: Sunday 06:00–08:00 AM exhibits a lower aggregate building power draw (~55 kW average, 51.9 kW at 07:00), but that is primarily because exterior and ground-floor lighting (`Floor1_z4_Light` and `Floor1_z1_Light`) shut off around sunrise, accounting for **~24 kW — essentially the entire net drop** (a further ~2.5 kW fall in ground-floor plug receptacles is offset by early indoor lighting switching on, e.g. `Floor1_z3_Light` +1.2 kW).
> * **Operational Transition Window**: The 06:00–08:00 AM window is an active operational transition period—exterior loads are switching off while early indoor lighting is starting to come on (likely early facilities, security, or janitorial activity). Because loads are actively changing states, it does not represent a steady-state baseline.
> * **Why Sunday 02:00–04:00 AM**: We chose Sunday 02:00–04:00 AM (~79.0 kW) because it is the **most stable, lowest-occupancy window**, capturing the steady-state load that runs continuously overnight, including timer-driven exterior lighting.
> * **Strategic Policy Implication**: Because our baseload captures this steady-state overnight profile, any potential curfew sweeps to exterior and atrium lighting appear as explicit, addressable policy savings opportunities (evaluated in Section 10) rather than being masked in the baseline.


## 4. Empirical Schedule Validation: Weekday vs. Saturday Load Profiles
To validate the assumed 08:00–17:00 occupied schedule against physical building behavior, we plot the mean hourly profile of workdays vs. Saturdays.


In [5]:
# ==============================================================================
# 4. SCHEDULE VALIDATION: WEEKDAY VS SATURDAY HOURLY PROFILE
# ==============================================================================
wkday_mask = (bldg_load.index.dayofweek < 5) & (~pd.Series(bldg_load.index.date, index=bldg_load.index).isin(holiday_dates))
sat_mask = bldg_load.index.dayofweek == 5

mean_wkday = bldg_load.loc[wkday_mask].groupby(bldg_load.loc[wkday_mask].index.hour).mean()
mean_sat = bldg_load.loc[sat_mask].groupby(bldg_load.loc[sat_mask].index.hour).mean()

fig, ax1 = plt.subplots(figsize=(13, 5))
ax1.plot(mean_wkday.index, mean_wkday.values, color='#0284C7', lw=2.5, marker='o', label='Weekday Mean Load (Mon-Fri)')
ax1.plot(mean_sat.index, mean_sat.values, color='#64748B', lw=2.0, ls='--', marker='s', label='Saturday Mean Load')
ax1.axvspan(8, 17, color='#0284C7', alpha=0.10, label='Core Occupied Window (08:00 - 17:00)')

# Net occupancy delta on secondary axis
ax2 = ax1.twinx()
delta = mean_wkday - mean_sat
ax2.plot(delta.index, delta.values, color='#F43F5E', lw=1.8, ls=':', label='Net Occupancy Delta (Weekday - Sat)')
ax2.set_ylabel("Net Occupancy Delta (kW)", color='#F43F5E', fontweight='bold')

ax1.set_xlabel("Hour of Day (00:00 - 23:00)", fontweight='bold')
ax1.set_ylabel("Active Power (kW)", fontweight='bold')
ax1.set_title("Chamchuri 5 24-Hour Daily Power Profile & Net Occupant Load: Weekday vs. Saturday Profiles", fontsize=12, fontweight='bold')
ax1.set_xticks(range(24))

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.show()



/var/folders/1g/x_wv4b8n0zs5kkhz0j1mg7v40000gn/T/ipykernel_3494/2215020937.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


> **Operational Schedule Boundary Validation Finding**:
> * **Daytime Surge**: Active power surges by an average of **+85.3 kW** between 08:00 and 17:00 on weekdays relative to Saturday.
> * **Evening Decline**: Load declines steadily after 17:00 (dropping to 105.5 kW at 22:00 and 88.5 kW at 23:00, converging on the ~79–80 kW overnight floor after midnight).
> * **Conclusion**: Mon–Fri 08:00–17:00 is stable across years as the occupied boundary; after 17:00 represents unoccupied curfew hours.


## 5. Mathematical Baseline Derivation: Dynamic Time-of-Week ($P_{2.5}$)

We derive the circuit baseload using the **2.5th percentile ($P_{2.5}, q=0.025$)**:
1. **Reference Quiescent Baseload ($B_{i,\text{ref}}$)**: Evaluated during the deep-sleep calibration window (Sundays 02:00–04:00 AM) across 52 training weeks. Used as the core feature for machine learning circuit clustering.
2. **Dynamic Time-of-Week Operational Baseload ($B_i(d, h)$)**: Evaluated for each of the unoccupied time-of-week slots across the training year. This accommodates natural operational departure transitions (e.g. 17:00–19:00 evening shoulder) while maintaining the deep-night floor during quiescent hours.


In [6]:
# ==============================================================================
# 5. BASELINE DERIVATION ACROSS ALL 65 CIRCUITS (STATIC + DYNAMIC TIME-OF-WEEK P2.5)
# ==============================================================================
BASELINE_PERCENTILE = 0.025
REF_DAY_OF_WEEK     = 6        # Sunday
REF_HOUR_START      = 2        # 02:00 AM
REF_HOUR_END        = 4        # 04:00 AM (exclusive: 02:00 to 04:00 AM)
pct_label = f"P{BASELINE_PERCENTILE * 100:g}"

circuit_baselines = {}
baseline_records = []

# 1. Calibrate Reference Deep-Sleep Window (Sundays 02:00-04:00) for Clustering
for floor_name, df in floor_dfs.items():
    train_df = df[df['Split'] == 'Train']
    ref_window = train_df[(train_df['DayOfWeek'] == REF_DAY_OF_WEEK) & 
                          (train_df['Hour'] >= REF_HOUR_START) & 
                          (train_df['Hour'] < REF_HOUR_END)]
    
    p_cols = [c for c in df.columns if ('_Light(kW)' in c) or ('_Plug(kW)' in c)]
    
    for col in p_cols:
        c_type = 'Lighting' if '_Light' in col else 'Plug Load'
        series = ref_window[col].dropna()
        b_val = float(series.quantile(BASELINE_PERCENTILE)) if len(series) > 0 else 0.0
        if np.isnan(b_val): b_val = 0.0
        
        cv = float(series.std() / (series.mean() + 1e-4)) if len(series) > 0 else 0.0
        circuit_id = f"{floor_name}_{col}"
        circuit_baselines[circuit_id] = b_val
        
        # Identify server room based on documented architectural layout (Salama 2021 Table 1)
        is_server = "Floor1_z2_Plug" in circuit_id
        
        baseline_records.append({
            'Circuit_ID': circuit_id,
            'Floor': floor_name,
            'Type': c_type,
            f'Baseline_kW_{pct_label}': round(b_val, 2),
            'Ref_Mean_kW': round(float(series.mean()), 2) if len(series) > 0 else 0.0,
            'CV': round(cv, 3),
            'Policy': 'Critical Standby (IT Server)' if is_server else 'Standard Curfew'
        })

baseline_df = pd.DataFrame(baseline_records)

# 2. Calibrate Dynamic Time-of-Week Baseload Matrix: B_i(DayOfWeek, Hour)
# Load hourly master dataset to compute unoccupied time-of-week quantiles
master_hourly = pd.read_csv(os.path.join(data_dir, 'qn1_hourly_master_for_excel.csv'), parse_dates=['Date'], index_col='Date')
train_unocc_h = master_hourly[(master_hourly['Is_Train_Period'] == 1) & (master_hourly['Is_Unoccupied'] == 1)]

all_circuit_cols = [c for c in master_hourly.columns if ('_Light(kW)' in c) or ('_Plug(kW)' in c)]
dynamic_base_matrix = train_unocc_h.groupby([train_unocc_h.index.dayofweek, train_unocc_h.index.hour])[all_circuit_cols].quantile(BASELINE_PERCENTILE)
dynamic_base_matrix.index.names = ['DayOfWeek', 'Hour']

# Export dynamic baseline matrix for digital twin synchronization
dyn_matrix_export_path = os.path.join(data_dir, 'qn1_dynamic_baseload_matrix.csv')
dynamic_base_matrix.reset_index().to_csv(dyn_matrix_export_path, index=False)
print(f"[OK] Dynamic Baseload Matrix calibrated across 168h and exported -> {os.path.basename(dyn_matrix_export_path)}")

display(baseline_df[['Circuit_ID', 'Floor', 'Type', f'Baseline_kW_{pct_label}', 'Ref_Mean_kW', 'CV', 'Policy']].head(10))


[OK] Dynamic Baseload Matrix calibrated across 168h and exported -> qn1_dynamic_baseload_matrix.csv


,Circuit_ID,Floor,Type,Baseline_kW_P2.5,Ref_Mean_kW,CV,Policy
0,Floor1_z1_Light(kW),Floor1,Lighting,0.00,11.03,0.602,Standard Curfew
1,Floor1_z1_Plug(kW),Floor1,Plug Load,0.04,4.38,1.499,Standard Curfew
2,Floor1_z2_Light(kW),Floor1,Lighting,4.23,11.24,0.397,Standard Curfew
3,Floor1_z2_Plug(kW),Floor1,Plug Load,8.39,15.45,0.190,Critical Standby (IT Server)
4,Floor1_z3_Light(kW),Floor1,Lighting,5.52,12.43,0.329,Standard Curfew
5,Floor1_z3_Plug(kW),Floor1,Plug Load,0.37,1.14,0.505,Standard Curfew
6,Floor1_z4_Light(kW),Floor1,Lighting,5.19,21.57,0.524,Standard Curfew
7,Floor2_z1_Light(kW),Floor2,Lighting,0.01,0.03,3.585,Standard Curfew
8,Floor2_z1_Plug(kW),Floor2,Plug Load,0.09,0.18,0.300,Standard Curfew
9,Floor2_z2_Light(kW),Floor2,Lighting,0.00,0.00,0.000,Standard Curfew


> **Mathematical Baseline Derivation Finding ($P_{2.5}$)**:
> * **Deep-Night Calibration ($P_{2.5}$)**: Quiescent floors were calibrated using the 2.5th percentile during the Sunday 02:00–04:00 AM window across 52 training weeks. The IT Server Room (`Floor1_z2_Plug`) establishes an empirical floor of **8.39 kW** with an off-hours CV of **0.190**.
> * **Dynamic Time-of-Week Operational Matrix ($P_{2.5}(d, h)$)**: Calibrated across all unoccupied `(DayOfWeek, Hour)` slots, establishing an achievable baseline that respects normal evening departure transitions while maintaining deep-night quiescent floors.


## 6. Deliverable 02: Unsupervised Circuit Classification (Critical Standby vs. Preventable Waste)

---

### Purpose & Machine Learning Rationale
We deploy **Unsupervised Machine Learning (K-Means)** across all 65 electrical sub-meters for data-driven tagging of each circuit as either **Critical Standby** or **Preventable Waste** based on their physical operating patterns.

### Engineered Behavioral Features
Each circuit's 18-month telemetry is mapped into a normalized 3-dimensional behavioral space:
1. **Base-Load Ratio ($\bar{P}_{\text{unoccupied}} / \bar{P}_{\text{occupied}}$)**: Measures sensitivity to human working hours ($<0.30$ for standard office loads, $\approx 1.00$ for continuous 24/7 loads, $\gg 1.00$ for inverted night facade lighting).
2. **Magnitude of Standby Floor ($B_i = P_{2.5}$ in kW)**: Derived from the Sunday 02:00–04:00 AM deep-night calibration window across 52 training weeks.
3. **Off-Hours Stability / Volatility ($CV_{\text{unoccupied}} = \sigma / \mu$)**: Relative fluctuation during unoccupied hours (steady machine draw vs. human switching).

### Deliverable Alignment
Each circuit is tagged as:
* **`Critical Standby`**: Continuous 24/7 mission-critical equipment (e.g. Server Room / IT lab in Floor 1 Zone 2) with high baseload floor and base-load ratio (~1.0).
* **`Preventable Waste`**: Curfew-controllable office lighting and computer plug circuits exhibiting human schedule dependence and unoccupied surplus.


In [7]:
# ==============================================================================
# 6A. DELIVERABLE 02: K-MEANS CIRCUIT CLASSIFICATION & DECISION BOUNDARY
# ==============================================================================
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples

# 1. Feature Extraction Across All 65 Circuits
circuit_features = []
for floor_name, df in floor_dfs.items():
    train_df = df[df['Split'] == 'Train']
    occ_df = train_df[train_df['Is_Occupied'] == 1]
    unocc_df = train_df[train_df['Is_Unoccupied'] == 1]
    p_cols = [c for c in df.columns if ('_Light(kW)' in c) or ('_Plug(kW)' in c)]
    
    for c in p_cols:
        circuit_id = f"{floor_name}_{c}"
        c_type = 'Lighting' if '_Light' in c else 'Plug Load'
        m_occ = occ_df[c].mean()
        m_unocc = unocc_df[c].mean()
        b_val = circuit_baselines.get(circuit_id, 0.0)
        cv_unocc = unocc_df[c].std() / (unocc_df[c].mean() + 1e-4)
        ratio = m_unocc / (m_occ + 1e-3)
        
        circuit_features.append({
            'Circuit_ID': circuit_id,
            'Floor': floor_name,
            'Zone_Col': c.replace('(kW)', ''),
            'Type': c_type,
            'Unocc_to_Occ_Ratio': ratio,
            'Baseload_kW': b_val,
            'OffHours_CV': cv_unocc
        })

feat_df = pd.DataFrame(circuit_features)
feature_cols = ['Unocc_to_Occ_Ratio', 'Baseload_kW', 'OffHours_CV']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(feat_df[feature_cols])

# 2. Multi-k Hyperparameter Benchmark (Inertia & Silhouette Score)
eval_records = []
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=20).fit(X_scaled)
    eval_records.append({
        'k': k,
        'Inertia': round(km.inertia_, 2),
        'Silhouette': round(silhouette_score(X_scaled, km.labels_), 3)
    })

eval_df = pd.DataFrame(eval_records)
OPTIMAL_K = int(eval_df.loc[eval_df['Silhouette'].idxmax(), 'k'])

# 3. Fit Model & Tag Circuits into 'Critical Standby' vs 'Preventable Waste'
km_opt = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=20).fit(X_scaled)
feat_df['Cluster'] = km_opt.labels_

# Identify critical standby cluster (highest mean baseload floor)
# Identify critical standby cluster (highest mean baseload floor: n=4 circuits)
crit_cluster = feat_df.groupby('Cluster')['Baseload_kW'].mean().idxmax()
feat_df['Classification'] = np.where(feat_df['Cluster'] == crit_cluster, 'Critical Standby', 'Preventable Waste')



n_critical = (feat_df['Classification'] == 'Critical Standby').sum()
n_waste = (feat_df['Classification'] == 'Preventable Waste').sum()


# 4. Visualization: Elbow, Silhouette, and Circuit Classification Scatter Plot
fig = plt.figure(figsize=(19, 5.5))

# Subplot 1: Elbow Curve
ax1 = plt.subplot(1, 3, 1)
ax1.plot(eval_df['k'], eval_df['Inertia'], marker='o', color='#2563EB', lw=2, markersize=7)
ax1.set_title('Elbow Method (Inertia vs. k)', fontweight='bold', fontsize=11)
ax1.set_xlabel('Number of Clusters (k)', fontsize=10)
ax1.set_ylabel('Inertia (WCSS)', fontsize=10)
ax1.grid(True, alpha=0.3)

# Subplot 2: Silhouette Score Curve
ax2 = plt.subplot(1, 3, 2)
ax2.plot(eval_df['k'], eval_df['Silhouette'], marker='s', color='#059669', lw=2, markersize=7)
ax2.axhline(0.70, color='red', ls='--', alpha=0.7, label='Strong Threshold (>0.70)')
sil_opt = eval_df.loc[eval_df['k'] == OPTIMAL_K, 'Silhouette'].values[0]
ax2.axvline(OPTIMAL_K, color='#F59E0B', ls=':', lw=2, label=f'Optimal k={OPTIMAL_K} (Sil: {sil_opt:.3f})')
ax2.set_title('Silhouette Score vs. k', fontweight='bold', fontsize=11)
ax2.set_xlabel('Number of Clusters (k)', fontsize=10)
ax2.set_ylabel('Silhouette Coefficient', fontsize=10)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

# Subplot 3: Circuit Classification Scatter Plot (Deliverable 02)
ax3 = plt.subplot(1, 3, 3)
color_map = {'Preventable Waste': '#D97706', 'Critical Standby': '#1E293B'}

for cat in ['Preventable Waste', 'Critical Standby']:
    sub = feat_df[feat_df['Classification'] == cat]
    bubble_sizes = 50 + (sub['OffHours_CV'] * 50)
    ax3.scatter(sub['Unocc_to_Occ_Ratio'], sub['Baseload_kW'],
                s=bubble_sizes, c=color_map[cat], label=f"{cat} (n={len(sub)})",
                alpha=0.8, edgecolors='black', linewidth=0.7)

# Annotate critical standby points
for _, row in feat_df[feat_df['Classification'] == 'Critical Standby'].iterrows():
    c_short = row['Circuit_ID'].replace('(kW)', '')
    ax3.annotate(c_short, (row['Unocc_to_Occ_Ratio'], row['Baseload_kW']),
                 textcoords='offset points', xytext=(6, 5), fontsize=8, fontweight='bold', color='#1E293B')

# Empirical cluster separation (K-Means separates the clusters on baseload magnitude, not on a fixed ratio threshold)
crit_min_b = feat_df.loc[feat_df['Classification'] == 'Critical Standby', 'Baseload_kW'].min()
prev_max_b = feat_df.loc[feat_df['Classification'] == 'Preventable Waste', 'Baseload_kW'].max()
sep_b = (crit_min_b + prev_max_b) / 2
ax3.axhline(y=sep_b, color='#DC2626', ls='--', lw=1.5, label=f'Cluster Separation (B = {sep_b:.1f} kW)')
ax3.set_title('Deliverable 02: Circuit Classification', fontweight='bold', fontsize=11)
ax3.set_xlabel('Base-Load Ratio (Unoccupied / Occupied)', fontsize=10)
ax3.set_ylabel('Standby Baseload Floor P2.5 (kW)', fontsize=10)
ax3.legend(fontsize=8, loc='upper right', frameon=True)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


/var/folders/1g/x_wv4b8n0zs5kkhz0j1mg7v40000gn/T/ipykernel_3494/940880056.py:120: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


> **Deliverable 02: Circuit Classification Findings**:
> * **Unsupervised K-Means Partition**: Using Base-Load Ratio, Baseload Floor, and Off-Hours CV, the data-driven algorithm isolates two physical clusters ($k=2$, Silhouette Score = 0.735):
>   * **Critical Standby (4 circuits)**: High-baseload, continuous operating profiles ($B_i > 4.0\text{ kW}$):
>     * `Floor1_z2_Plug(kW)` — IT Server Data Center (8.39 kW baseline, Ratio = 1.01)
>     * `Floor1_z2_Light(kW)` — 24/7 Corridor / Infrastructure Lighting (4.23 kW baseline)
>     * `Floor1_z3_Light(kW)` — Continuous Zone 3 Lighting (5.52 kW baseline)
>     * `Floor1_z4_Light(kW)` — Night Exterior Facade Lighting (5.19 kW baseline)
>   * **Preventable Waste (61 circuits)**: Standard office lighting and computer plug circuits exhibiting low baseload floors (~0.03 kW) and high schedule dependence, targetable by BEMS curfew sweeps.


## 6B. Behaviour-Based Circuit Classification (Scale-Free Features)

**Why this section exists.** An ablation test on the Section 6 clustering shows that its result is decided by `Baseload_kW` alone: removing `Unocc_to_Occ_Ratio` or `OffHours_CV` gives identical clusters (ARI = 1.0). Because `Baseload_kW` is measured in absolute kW, K-Means separated **big** circuits from **small** ones, not **always-needed** circuits from **switch-off-able** ones. As a result, the Floor 1 night/facade lighting was labelled *Critical Standby*.

Here we describe each circuit **relative to itself**, so a 0.3 kW stair-hall plug circuit and the 19 kW electrical engine room look alike if they both run around the clock. All features use the **training year only** (Jul 2018 – Jun 2019).

| Feature | Question it answers | Critical Standby (24/7) | Preventable (follows occupancy) |
| :--- | :--- | :--- | :--- |
| `Night_On_Frac` | Share of after-hours hours the circuit is clearly ON (> 20 % of its own peak) | ≈ 1.0 | low (≈ 0.1–0.3) |
| `Base_to_Peak` | Overnight floor ($P_{2.5}$ of after-hours load) ÷ its own peak ($P_{95}$) | high: barely drops | ≈ 0: drops to nothing |
| `Night_to_Day` | Mean load 19:00–05:00 ÷ mean occupied-hours load (log) | ≈ 1 | ≪ 1 (facade/timer lights: ≫ 1) |
| `Weekend_to_Weekday` | Weekend 09:00–17:00 mean ÷ weekday occupied mean (log) | ≈ 1 | ≪ 1 |
| `OffHours_CV` | Volatility of after-hours load (σ/μ) | low (steady machine) | high (human switching) |

The full justification is in [`qn1_kmeans_feature_justification.md`](qn1_kmeans_feature_justification.md).

In [8]:
# ==============================================================================
# 6B-1. SCALE-FREE BEHAVIOUR FEATURES (TRAINING YEAR ONLY)
# ==============================================================================
PEAK_Q       = 0.95      # a circuit's own "typical peak" = 95th percentile of its hourly load
ON_THRESHOLD = 0.20      # "clearly ON" = above 20% of the circuit's own peak
NIGHT_START, NIGHT_END = 19, 5   # night window 19:00-05:00
DAY_START, DAY_END     = 9, 17   # daytime window for the weekend comparison
RATIO_FLOOR  = 0.01      # floor for ratios before taking logs

BEHAVIOUR_FEATURES = ['Night_On_Frac', 'Base_to_Peak', 'Night_to_Day', 'Weekend_to_Weekday', 'OffHours_CV']
LOG_FEATURES = ['Night_to_Day', 'Weekend_to_Weekday']

def behaviour_features(hourly, circuits):
    """One row per circuit; every feature is relative to the circuit's own load level."""
    h, dow = hourly.index.hour, hourly.index.dayofweek
    occ   = (hourly['Is_Occupied'] == 1).to_numpy()
    unocc = (hourly['Is_Unoccupied'] == 1).to_numpy()
    night = (h >= NIGHT_START) | (h < NIGHT_END)
    wkend_day = (dow >= 5) & (h >= DAY_START) & (h < DAY_END)
    rows = []
    for c in circuits:
        s = hourly[c]
        peak = s.quantile(PEAK_Q)
        s_unocc = s[unocc].dropna()
        occ_mean = s[occ].mean()
        rows.append({
            'Circuit_ID': c,
            'Night_On_Frac': float((s_unocc > ON_THRESHOLD * peak).mean()),
            'Base_to_Peak': float(s_unocc.quantile(BASELINE_PERCENTILE) / peak) if peak > 0 else 0.0,
            'Night_to_Day': float(s[night].mean() / (occ_mean + 1e-3)),
            'Weekend_to_Weekday': float(s[wkend_day].mean() / (occ_mean + 1e-3)),
            'OffHours_CV': float(s_unocc.std() / (s_unocc.mean() + 1e-4)),
            'Peak_kW': float(peak),
        })
    return pd.DataFrame(rows).set_index('Circuit_ID')

def to_model_matrix(feat):
    """Log-transform the two ratio features (they span 0.02 to 30), leave the bounded ones as they are."""
    X = feat[BEHAVIOUR_FEATURES].copy()
    X[LOG_FEATURES] = np.log(X[LOG_FEATURES].clip(lower=RATIO_FLOOR))
    return X

hourly_train = master_hourly[master_hourly['Is_Train_Period'] == 1]
beh_train = behaviour_features(hourly_train, all_circuit_cols)
X_beh_raw = to_model_matrix(beh_train)

# Skewness before vs after the log transform (K-Means uses Euclidean distance, so heavy skew lets a few circuits dominate)
skew_df = pd.DataFrame({'Skew_Raw': beh_train[BEHAVIOUR_FEATURES].skew(),
                        'Skew_Used_In_Model': X_beh_raw.skew()}).round(2)
print('=== FEATURE SKEWNESS (raw vs. as used in K-Means) ===')
display(skew_df)

print('=== SPEARMAN CORRELATION BETWEEN FEATURES (|rho| < 0.9 -> no feature is a duplicate) ===')
display(beh_train[BEHAVIOUR_FEATURES].corr(method='spearman').round(2))

=== FEATURE SKEWNESS (raw vs. as used in K-Means) ===


,Skew_Raw,Skew_Used_In_Model
Night_On_Frac,1.31,1.31
Base_to_Peak,3.27,3.27
Night_to_Day,5.99,1.25
Weekend_to_Weekday,1.35,0.13
OffHours_CV,0.10,0.10


=== SPEARMAN CORRELATION BETWEEN FEATURES (|rho| < 0.9 -> no feature is a duplicate) ===


,Night_On_Frac,Base_to_Peak,Night_to_Day,Weekend_to_Weekday,OffHours_CV
Night_On_Frac,1.00,0.42,0.72,0.87,-0.59
Base_to_Peak,0.42,1.00,0.68,0.39,-0.83
Night_to_Day,0.72,0.68,1.00,0.77,-0.85
Weekend_to_Weekday,0.87,0.39,0.77,1.00,-0.59
OffHours_CV,-0.59,-0.83,-0.85,-0.59,1.00


In [9]:
# ==============================================================================
# 6B-2. CHOOSE k, FIT K-MEANS, AND NAME THE CLUSTERS BY BEHAVIOUR
# ==============================================================================
beh_scaler = StandardScaler().fit(X_beh_raw)
X_beh = beh_scaler.transform(X_beh_raw)

k_rows = []
for k in range(2, 7):
    lab = KMeans(n_clusters=k, random_state=42, n_init=50).fit(X_beh).labels_
    k_rows.append({'k': k, 'Silhouette': round(silhouette_score(X_beh, lab), 3),
                   'Cluster_Sizes': sorted(np.bincount(lab).tolist(), reverse=True)})
print('=== CANDIDATE k ===')
display(pd.DataFrame(k_rows))

# k = 3 is the smallest k that separates the three physical behaviours (see markdown below):
# k = 2 merges the 24/7 loads with the night/extended-hours lighting, which is the error we are fixing.
BEHAVIOUR_K = 3
km_beh = KMeans(n_clusters=BEHAVIOUR_K, random_state=42, n_init=50).fit(X_beh)
beh_train['Cluster'] = km_beh.labels_

# Name clusters from their medians, so names do not depend on K-Means' arbitrary label numbers
GROUP_CRITICAL = 'Critical Standby (24/7 Steady)'
GROUP_EXTENDED = 'Extended-Hours / Scheduled'
GROUP_OCCUPANCY = 'Occupancy-Following'
cl_med = beh_train.groupby('Cluster')[BEHAVIOUR_FEATURES].median()
crit_id = cl_med['Base_to_Peak'].idxmax()
occ_id = cl_med['Night_On_Frac'].idxmin()
ext_ids = [c for c in cl_med.index if c not in (crit_id, occ_id)]
assert crit_id != occ_id and len(ext_ids) == 1, 'Cluster naming is ambiguous; inspect cl_med'
cluster_names = {crit_id: GROUP_CRITICAL, occ_id: GROUP_OCCUPANCY, ext_ids[0]: GROUP_EXTENDED}
beh_train['Behaviour_Group'] = beh_train['Cluster'].map(cluster_names)
group_order = [GROUP_CRITICAL, GROUP_EXTENDED, GROUP_OCCUPANCY]

profile = beh_train.groupby('Behaviour_Group')[BEHAVIOUR_FEATURES].median().reindex(group_order).round(2)
profile['Circuits'] = beh_train['Behaviour_Group'].value_counts().reindex(group_order)
print(f'=== CLUSTER PROFILES (median feature values), k={BEHAVIOUR_K}, silhouette={silhouette_score(X_beh, km_beh.labels_):.3f} ===')
display(profile)

for g in [GROUP_CRITICAL, GROUP_EXTENDED]:
    members = beh_train.index[beh_train['Behaviour_Group'] == g].str.replace('(kW)', '', regex=False)
    print(f'{g} ({len(members)}): {", ".join(members)}')

print('\n=== NEW BEHAVIOUR GROUPS vs. SECTION 6 CLASSIFICATION ===')
display(pd.crosstab(beh_train['Behaviour_Group'],
                    feat_df.set_index('Circuit_ID').loc[beh_train.index, 'Classification']).reindex(group_order))

=== CANDIDATE k ===


,k,Silhouette,Cluster_Sizes
0,2,0.511,"[49, 16]"
1,3,0.426,"[43, 16, 6]"
2,4,0.351,"[27, 22, 11, 5]"
3,5,0.367,"[27, 22, 8, 4, 4]"
4,6,0.373,"[22, 17, 12, 6, 4, 4]"


=== CLUSTER PROFILES (median feature values), k=3, silhouette=0.426 ===


,Night_On_Frac,Base_to_Peak,Night_to_Day,Weekend_to_Weekday,OffHours_CV,Circuits
Behaviour_Group,,,,,,
Critical Standby (24/7 Steady),1.00,0.58,0.96,0.96,0.16,6
Extended-Hours / Scheduled,0.53,0.04,0.44,0.43,0.74,16
Occupancy-Following,0.21,0.00,0.07,0.15,1.62,43


Critical Standby (24/7 Steady) (6): Floor1_z2_Light, Floor1_z2_Plug, Floor2_z3_Plug, Floor3_z3_Plug, Floor5_z3_Plug, Floor6_z3_Plug
Extended-Hours / Scheduled (16): Floor1_z1_Light, Floor1_z1_Plug, Floor1_z3_Light, Floor1_z3_Plug, Floor1_z4_Light, Floor2_z3_Light, Floor2_z4_Plug, Floor4_z2_Plug, Floor4_z3_Light, Floor4_z3_Plug, Floor4_z5_Plug, Floor6_z1_Plug, Floor6_z3_Light, Floor7_z3_Light, Floor7_z3_Plug, Floor7_z5_Plug

=== NEW BEHAVIOUR GROUPS vs. SECTION 6 CLASSIFICATION ===


Classification,Critical Standby,Preventable Waste
Behaviour_Group,,
Critical Standby (24/7 Steady),2,4
Extended-Hours / Scheduled,2,14
Occupancy-Following,0,43


In [10]:
# ==============================================================================
# 6B-3. DOES EVERY FEATURE MATTER? ABLATION + STABILITY CHECKS
# ==============================================================================
from sklearn.metrics import adjusted_rand_score

def _align(ref, lab):
    """Relabel `lab` so each of its clusters takes the reference label it overlaps most."""
    ct = pd.crosstab(lab, ref)
    return np.array([ct.loc[l].idxmax() for l in lab])

def ablation(X, names, k, full_labels, drop_sets):
    """For each set of features removed: ARI vs the full clustering, and how many circuits change group.
    ARI = 1.0 and 0 moved -> the removed feature(s) contribute nothing."""
    rows = []
    for drop in drop_sets:
        keep = [i for i, n in enumerate(names) if n not in drop]
        lab = KMeans(n_clusters=k, random_state=42, n_init=50).fit(X[:, keep]).labels_
        rows.append({'Feature_Removed': ' + '.join(drop),
                     'ARI_vs_Full': round(adjusted_rand_score(full_labels, lab), 2),
                     'Circuits_Moved': int((_align(full_labels, lab) != full_labels).sum())})
    return pd.DataFrame(rows)

abl_old = ablation(X_scaled, feature_cols, OPTIMAL_K, km_opt.labels_, [[n] for n in feature_cols])
abl_new = ablation(X_beh, BEHAVIOUR_FEATURES, BEHAVIOUR_K, km_beh.labels_,
                   [[n] for n in BEHAVIOUR_FEATURES] + [['Weekend_to_Weekday', 'OffHours_CV']])
ablation_df = pd.concat([abl_old.assign(Model='Section 6 (old features)'),
                         abl_new.assign(Model='Section 6B (behaviour features)')], ignore_index=True)
ablation_df['Contribution'] = np.select(
    [ablation_df['Circuits_Moved'] == 0, ablation_df['ARI_vs_Full'] >= 0.9],
    ['None (redundant)', 'Weak (tie-breaker)'], 'Strong')
ablation_df = ablation_df[['Model', 'Feature_Removed', 'ARI_vs_Full', 'Circuits_Moved', 'Contribution']]
print('=== ABLATION: ARI between full clustering and clustering without each feature ===')
display(ablation_df)

# Seed stability: K-Means starts from random centroids; the grouping should not depend on the seed
seed_ari = [adjusted_rand_score(km_beh.labels_, KMeans(n_clusters=BEHAVIOUR_K, random_state=s, n_init=50).fit(X_beh).labels_)
            for s in range(10)]
print(f'Seed stability: minimum ARI across 10 random seeds = {min(seed_ari):.3f}')

# Out-of-sample stability: recompute features on Jul-Dec 2019 and assign each circuit to the TRAINED centroids
hourly_test = master_hourly[master_hourly.index >= '2019-07-01']
beh_test = behaviour_features(hourly_test, all_circuit_cols)
test_groups = pd.Series(km_beh.predict(beh_scaler.transform(to_model_matrix(beh_test))), index=beh_test.index).map(cluster_names)
same = (test_groups == beh_train['Behaviour_Group'])
print(f'Out-of-sample stability: {same.mean():.1%} of circuits keep the same group on Jul-Dec 2019 data')
moved = pd.DataFrame({'Train_Group': beh_train.loc[~same, 'Behaviour_Group'], 'Test_Group': test_groups[~same]})
if len(moved):
    display(moved)

=== ABLATION: ARI between full clustering and clustering without each feature ===


,Model,Feature_Removed,ARI_vs_Full,Circuits_Moved,Contribution
0,Section 6 (old features),Unocc_to_Occ_Ratio,1.00,0,None (redundant)
1,Section 6 (old features),Baseload_kW,0.28,4,Strong
2,Section 6 (old features),OffHours_CV,1.00,0,None (redundant)
3,Section 6B (behaviour features),Night_On_Frac,0.54,10,Strong
4,Section 6B (behaviour features),Base_to_Peak,0.34,16,Strong
5,Section 6B (behaviour features),Night_to_Day,0.29,16,Strong
6,Section 6B (behaviour features),Weekend_to_Weekday,0.98,1,Weak (tie-breaker)
7,Section 6B (behaviour features),OffHours_CV,0.98,1,Weak (tie-breaker)
8,Section 6B (behaviour features),Weekend_to_Weekday + OffHours_CV,0.87,3,Strong


Seed stability: minimum ARI across 10 random seeds = 0.944
Out-of-sample stability: 89.2% of circuits keep the same group on Jul-Dec 2019 data


,Train_Group,Test_Group
Circuit_ID,,
Floor3_z3_Light(kW),Occupancy-Following,Extended-Hours / Scheduled
Floor4_z3_Plug(kW),Extended-Hours / Scheduled,Critical Standby (24/7 Steady)
Floor4_z4_Light(kW),Occupancy-Following,Extended-Hours / Scheduled
Floor5_z4_Plug(kW),Occupancy-Following,Extended-Hours / Scheduled
Floor6_z5_Plug(kW),Occupancy-Following,Extended-Hours / Scheduled
Floor7_z1_Plug(kW),Occupancy-Following,Extended-Hours / Scheduled
Floor7_z3_Plug(kW),Extended-Hours / Scheduled,Critical Standby (24/7 Steady)


In [11]:
# ==============================================================================
# 6B-4. DO THE GROUPS BEHAVE DIFFERENTLY? NORMALISED 24-HOUR PROFILES + FEATURE MAP
# ==============================================================================
GROUP_COLORS = {GROUP_CRITICAL: '#2a78d6', GROUP_EXTENDED: '#eb6834', GROUP_OCCUPANCY: '#1baf7a'}

# Each circuit's mean load by hour of day (weekdays, training year), divided by its own P95 peak
wk = hourly_train[hourly_train.index.dayofweek < 5]
norm_profiles = wk[all_circuit_cols].groupby(wk.index.hour).mean() / beh_train['Peak_kW'].replace(0, np.nan)

fig, (ax_p, ax_s) = plt.subplots(1, 2, figsize=(16, 5.5))
for g in group_order:
    members = beh_train.index[beh_train['Behaviour_Group'] == g]
    med = norm_profiles[members].median(axis=1)
    ax_p.plot(med.index, med.values, color=GROUP_COLORS[g], lw=2, label=f'{g} (n={len(members)})')
    ax_p.text(23.3, med.values[-1], g.split(' (')[0], color='#333333', fontsize=9, va='center')
ax_p.axvspan(8, 17, color='#888888', alpha=0.08)
ax_p.set_xticks(range(0, 24, 2)); ax_p.set_xlim(0, 28)
ax_p.set_xlabel('Hour of day (weekdays)', fontweight='bold')
ax_p.set_ylabel('Load as a share of the circuit\'s own peak', fontweight='bold')
ax_p.set_title('Median normalised weekday profile by behaviour group', fontweight='bold')
ax_p.legend(loc='upper left', fontsize=9)

for g in group_order:
    sub = beh_train[beh_train['Behaviour_Group'] == g]
    ax_s.scatter(sub['Night_On_Frac'], sub['Base_to_Peak'], s=60, color=GROUP_COLORS[g],
                 edgecolors='white', linewidth=1.5, label=g)
label_offsets = {'Floor1_z1_Light(kW)': (-20, 10), 'Floor1_z4_Light(kW)': (0, -14)}
for cid, row in beh_train[beh_train.index.str.startswith('Floor1_')].iterrows():
    if row['Behaviour_Group'] != GROUP_OCCUPANCY:
        ax_s.annotate(cid.replace('(kW)', ''), (row['Night_On_Frac'], row['Base_to_Peak']),
                      textcoords='offset points', xytext=label_offsets.get(cid, (6, 4)), fontsize=8, color='#333333')
ax_s.set_xlabel('Night_On_Frac (share of after-hours time clearly ON)', fontweight='bold')
ax_s.set_ylabel('Base_to_Peak (overnight floor / own peak)', fontweight='bold')
ax_s.set_title('Two of the five features: groups separate without using kW', fontweight='bold')
ax_s.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

/var/folders/1g/x_wv4b8n0zs5kkhz0j1mg7v40000gn/T/ipykernel_3494/2649061635.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
# ==============================================================================
# 6B-5. IMPACT ON THE WASTE ESTIMATE + OPTIONAL SWITCH TO USE THE NEW CLASSIFICATION
# ==============================================================================
# False keeps the Section 6 labels for Sections 7-12 (headline numbers unchanged).
# True protects only the 24/7 group; everything else becomes Preventable Waste. Re-run all cells below
# and update the hard-coded markdown findings and the webapp text if you switch this on.
APPLY_BEHAVIOUR_CLASSIFICATION = False

new_class = np.where(beh_train['Behaviour_Group'] == GROUP_CRITICAL, 'Critical Standby', 'Preventable Waste')
new_class = pd.Series(new_class, index=beh_train.index)
feat_df['Behaviour_Group'] = feat_df['Circuit_ID'].map(beh_train['Behaviour_Group'])

# Theoretical surplus above the dynamic P2.5 baseline, training year, per circuit
_un = hourly_train[hourly_train['Is_Unoccupied'] == 1]
_base = dynamic_base_matrix.reindex(pd.MultiIndex.from_arrays([_un.index.dayofweek, _un.index.hour]))[all_circuit_cols].to_numpy()
_excess = pd.Series(np.nansum(np.clip(_un[all_circuit_cols].to_numpy() - _base, 0, None), axis=0), index=all_circuit_cols)
_after_hours = _un[all_circuit_cols].sum()

old_class = feat_df.set_index('Circuit_ID')['Classification']
impact_df = pd.DataFrame([
    {'Classification': 'Section 6 (old features)', 'Critical_Standby': int((old_class == 'Critical Standby').sum()),
     'Preventable_Waste': int((old_class == 'Preventable Waste').sum()),
     'Theoretical_Waste_MWh_yr': round(_excess[old_class.index[old_class == 'Preventable Waste']].sum() / 1000, 1)},
    {'Classification': 'Section 6B (behaviour features)', 'Critical_Standby': int((new_class == 'Critical Standby').sum()),
     'Preventable_Waste': int((new_class == 'Preventable Waste').sum()),
     'Theoretical_Waste_MWh_yr': round(_excess[new_class.index[new_class == 'Preventable Waste']].sum() / 1000, 1)},
])
print('=== IMPACT ON THE TRAINING-YEAR WASTE ESTIMATE (surplus above dynamic P2.5 baseline) ===')
display(impact_df)

grp_energy = pd.DataFrame({'After_Hours_MWh': _after_hours.groupby(beh_train['Behaviour_Group']).sum() / 1000,
                           'Surplus_Above_Baseline_MWh': _excess.groupby(beh_train['Behaviour_Group']).sum() / 1000}).reindex(group_order).round(1)
print('=== AFTER-HOURS ENERGY BY BEHAVIOUR GROUP (training year) ===')
display(grp_energy)

if APPLY_BEHAVIOUR_CLASSIFICATION:
    feat_df['Classification'] = feat_df['Circuit_ID'].map(new_class)
    print('[APPLIED] Sections 7-12 now use the behaviour-based classification.')
else:
    print('[INFO] Sections 7-12 still use the Section 6 classification. Set APPLY_BEHAVIOUR_CLASSIFICATION = True to switch.')

=== IMPACT ON THE TRAINING-YEAR WASTE ESTIMATE (surplus above dynamic P2.5 baseline) ===


,Classification,Critical_Standby,Preventable_Waste,Theoretical_Waste_MWh_yr
0,Section 6 (old features),4,61,175.7
1,Section 6B (behaviour features),6,59,354.3


=== AFTER-HOURS ENERGY BY BEHAVIOUR GROUP (training year) ===


,After_Hours_MWh,Surplus_Above_Baseline_MWh
Behaviour_Group,,
Critical Standby (24/7 Steady),183.5,71.1
Extended-Hours / Scheduled,380.8,262.6
Occupancy-Following,109.1,91.6


[INFO] Sections 7-12 still use the Section 6 classification. Set APPLY_BEHAVIOUR_CLASSIFICATION = True to switch.


> **Behaviour-Based Classification Findings (training year, k = 3)**:
> * **Every feature now contributes.** In Section 6, removing `Unocc_to_Occ_Ratio` or `OffHours_CV` left the clusters unchanged (ARI = 1.0). In Section 6B, removing `Night_On_Frac`, `Base_to_Peak` or `Night_to_Day` moves 10–16 circuits (ARI 0.29–0.54). `Weekend_to_Weekday` and `OffHours_CV` are tie-breakers: each moves one circuit alone (`Floor1_z2_Light`), and three together (ARI 0.87).
> * **Why k = 3 and not k = 2.** k = 2 has the higher silhouette (0.51 vs 0.43), but it puts the electrical engine room in the same group as the Floor 1 exterior lighting, which is the error this section fixes. k = 3 is the smallest k that separates the three behaviours the question needs: *must stay on*, *runs into the night on a schedule*, *follows people*. The lower silhouette is expected: five features that describe behaviour on continuous scales do not form the artificially "clean" gap that one extreme kW feature did.
> * **Critical Standby (24/7 steady, 6 circuits)**: the Floor 1 Zone 2 electrical engine room (plugs and lighting), plus four small (~0.3–0.6 kW) plug circuits in the Zone 3 central stair hall on Floors 2, 3, 5 and 6 that never switch off. What is connected to them is not documented; confirm with facilities. These were previously labelled Preventable Waste only because they are small.
> * **Extended-Hours / Scheduled (16 circuits)**: includes `Floor1_z4_Light` ("around the building", i.e. exterior lighting) and `Floor1_z1_Light` (sitting hall), with night-to-day ratios of about 32 and 19, and `Floor1_z3_Light` (central stair hall, which may need dimmed rather than switched-off lighting for safe exit routes). These are **schedulable**, not critical. They should be addressed with explicit schedule scenarios (e.g. facade off at 00:00) rather than protected at 0 waste.
> * **Occupancy-Following (43 circuits)**: upper-floor office lighting and desk plugs whose load drops to ~0 % of peak at night.
> * **Stability**: minimum ARI across 10 random seeds is 0.94, and about 89 % of circuits keep the same group when features are recomputed on the unseen Jul–Dec 2019 data.
> * **Impact**: treating only the 24/7 group as protected raises the *theoretical* training-year surplus from 175.7 to 354.3 MWh, almost all from the Floor 1 night lighting. Because this is a theoretical maximum, the Extended-Hours group should be reported with schedule scenarios before it changes the headline. Values above are printed by the cells in this section; re-check them after any re-run.

### Standby Time-Series Verification: Server Room vs. Office Lighting
We verify that `Floor1_z2_Plug` (the Smart Grid Research Unit living lab / server room, *Salama 2021 Table 1*) exhibits 24/7 continuous computing draw (8.39 kW baseline, $CV \approx 0.190$) and has been classified via a data-driven K-Means partition into the critical standby cluster.

In [13]:
# ==============================================================================
# 6. VISUALIZE SERVER STANDBY VS OFFICE LIGHTING
# ==============================================================================
f1_df = floor_dfs['Floor1']
f2_df = floor_dfs['Floor2']
sample_range_f1 = f1_df.loc['2019-02-15 00:00:00':'2019-02-25 00:00:00']
sample_range_f2 = f2_df.loc['2019-02-15 00:00:00':'2019-02-25 00:00:00']

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Subplot 1: Server Room Plug (Floor 1 Zone 2)
axes[0].plot(sample_range_f1.index, sample_range_f1['z2_Plug(kW)'], color='#2563EB', lw=1.8, label='Server Room Plug Load (Floor 1 z2_Plug)')
axes[0].axhline(circuit_baselines['Floor1_z2_Plug(kW)'], color='black', ls='--', lw=1.5, label=f'Server Baseline ({circuit_baselines["Floor1_z2_Plug(kW)"]:.2f} kW)')
axes[0].set_ylabel('Active Power (kW)', color='#2563EB', fontweight='bold')
axes[0].set_title('Floor 1 Zone 2: Continuous Server Standby (SGRU Living Lab — Critical Standby)', fontweight='bold')
axes[0].legend(loc='upper right')

# Subplot 2: Typical Upper-Floor Office Lighting (Floor 2 Zone 1)
axes[1].plot(sample_range_f2.index, sample_range_f2['z1_Light(kW)'], color='#059669', lw=1.8, label='Office Lighting (Floor 2 z1_Light)')
axes[1].axhline(circuit_baselines.get('Floor2_z1_Light(kW)', 0.0), color='black', ls='--', lw=1.5, label='Lighting Baseline (0.02 kW)')
axes[1].set_ylabel('Active Power (kW)', color='#059669', fontweight='bold')
axes[1].set_title('Floor 2 Zone 1: Typical Office Lighting (True Zero Off-Hours, Standard Daytime Office Operation)', fontweight='bold')
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()


/var/folders/1g/x_wv4b8n0zs5kkhz0j1mg7v40000gn/T/ipykernel_3494/4225500631.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Dynamic Addressable Schedule Surplus Integration

For all circuits classified as **Preventable Waste**, active power during unoccupied hours exceeding the dynamic Time-of-Week baseline is integrated:
$$W_i(t) = \max\Big(0,\, P_i(t) - B_i(d(t), h(t))\Big) \quad \text{for } i \in \text{Preventable Waste}$$

Circuits classified as **Critical Standby** (4 circuits, including IT Server Room and continuous lobby infrastructure) are protected essential loads and allocated **zero preventable waste** ($W_i = 0$).


In [14]:
# ==============================================================================
# 7. INTEGRATE ADDRESSABLE SURPLUS ACROSS ALL CIRCUITS (DYNAMIC TIME-OF-WEEK P2.5)
# ==============================================================================
# Dynamically query all Critical Standby circuits from Step 6 (Deliverable 02)
critical_standby_circuits = set(feat_df.loc[feat_df['Classification'] == 'Critical Standby', 'Circuit_ID'])

circuit_waste_records = []
floor_waste_totals = {f'Floor{i}': 0.0 for i in range(1, 8)}
type_waste_totals = {'Lighting': 0.0, 'Plug Load': 0.0}

# 18-Month Unoccupied Telemetry
unocc_master = master_hourly[master_hourly['Is_Unoccupied'] == 1].copy()
unocc_dow = unocc_master.index.dayofweek
unocc_hr = unocc_master.index.hour

circuit_excess_18m = {}
train_unocc_mask = (unocc_master['Is_Train_Period'] == 1).to_numpy()   # training year Jul 2018 - Jun 2019
floor_annual_totals = {f'Floor{i}': 0.0 for i in range(1, 8)}

for col in all_circuit_cols:
    circuit_id = col
    floor_name = col.split('_')[0]
    zone_col = col.replace(f"{floor_name}_", "").replace("(kW)", "")
    c_type = 'Lighting' if '_Light' in col else 'Plug Load'
    ref_b = circuit_baselines.get(circuit_id, 0.0)
    
    if circuit_id in critical_standby_circuits:
        excess_series = pd.Series(0.0, index=unocc_master.index)
        wasted_18m_kwh = 0.0
    else:
        # Vectorized lookup of dynamic baseline B_i(dow, hour)
        slot_keys = list(zip(unocc_dow, unocc_hr))
        base_series = pd.Series([dynamic_base_matrix.loc[k, col] if k in dynamic_base_matrix.index else ref_b 
                                 for k in slot_keys], index=unocc_master.index)
        excess_series = (unocc_master[col] - base_series).clip(lower=0.0)
        wasted_18m_kwh = float(excess_series.sum()) # hourly kW sum = kWh
        
    circuit_excess_18m[circuit_id] = excess_series
    # Annual figure = actual surplus in the 12-month training year (same basis as the Section 10 headline)
    annual_kwh = float(excess_series[train_unocc_mask].sum())
    
    circuit_waste_records.append({
        'Circuit_ID': circuit_id,
        'Floor': floor_name,
        'Zone_Col': zone_col,
        'Type': c_type,
        'Baseload_kW': round(ref_b, 2),
        'Dynamic_Baseload_Mean_kW': round(float(dynamic_base_matrix[col].mean()), 3),
        'Unocc_Mean_kW': round(float(unocc_master[col].mean()), 2),
        'Surplus_kWh_18M': round(wasted_18m_kwh, 1),
        'Annual_Surplus_kWh': round(annual_kwh, 1)
    })
    
    floor_waste_totals[floor_name] += wasted_18m_kwh
    floor_annual_totals[floor_name] += annual_kwh
    type_waste_totals[c_type] += wasted_18m_kwh

surplus_summary_df = pd.DataFrame(circuit_waste_records)
total_18m_kwh = surplus_summary_df['Surplus_kWh_18M'].sum()
annual_waste_12m_clean = sum(floor_annual_totals.values())

floor_waste_df = pd.DataFrame([{
    'Floor': k,
    'Preventable_Waste_MWh': round(v / 1000.0, 2),
    'Annual_Waste_MWh': round(floor_annual_totals[k] / 1000.0, 2)
} for k, v in floor_waste_totals.items()])

# Comparison: the same surplus measured against the static Sunday 02:00-04:00 baseline (training year)
_train_unocc = unocc_master[train_unocc_mask]
static_annual_kwh = sum(float((_train_unocc[c] - circuit_baselines.get(c, 0.0)).clip(lower=0.0).sum())
                        for c in all_circuit_cols if c not in critical_standby_circuits)

print(f"=== DELIVERABLE 01: DYNAMIC PREVENTABLE SURPLUS SUMMARY ===")
print(f"  18-Month Total Preventable Surplus: {total_18m_kwh:,.1f} kWh ({total_18m_kwh/1000:,.1f} MWh)")
print(f"  Annual Surplus (training year)    : {annual_waste_12m_clean:,.1f} kWh / year ({annual_waste_12m_clean/1000:,.1f} MWh/yr)")
print(f"  Same, static Sunday baseline      : {static_annual_kwh/1000:,.1f} MWh/yr "
      f"(dynamic baseline allows {(static_annual_kwh - annual_waste_12m_clean)/1000:,.1f} MWh/yr of normal evening load)")
display(floor_waste_df)


=== DELIVERABLE 01: DYNAMIC PREVENTABLE SURPLUS SUMMARY ===
  18-Month Total Preventable Surplus: 266,310.6 kWh (266.3 MWh)
  Annual Surplus (training year)    : 175,653.6 kWh / year (175.7 MWh/yr)
  Same, static Sunday baseline      : 200.4 MWh/yr (dynamic baseline allows 24.8 MWh/yr of normal evening load)


,Floor,Preventable_Waste_MWh,Annual_Waste_MWh
0,Floor1,107.50,72.79
1,Floor2,17.87,10.99
2,Floor3,24.49,14.85
3,Floor4,26.21,16.58
4,Floor5,28.37,18.24
5,Floor6,29.91,22.14
6,Floor7,31.95,20.06


> **Dynamic Addressable Surplus Integration Finding**:
> * **18-Month Cumulative Waste**: **266,311 kWh (266.3 MWh)** across all 61 preventable circuits under the dynamic Time-of-Week baseline.
> * **Annual Waste (training year, Jul 2018 – Jun 2019)**: **175,654 kWh / year** (with all 4 Critical Standby circuits allocated 0 waste). This is the same basis as the Section 10 headline.
> * **Departure Transition Accommodation**: Compared to the static Sunday baseline (200.4 MWh/yr), the dynamic baseline accommodates ~24.8 MWh/yr of natural evening departures and janitorial sweeps, establishing a realistic, achievable target for facility automation.


## 8. Forensic Circuit Audit: The 17:00–22:00 Evening Window
We investigate which circuits drive the 17:00–22:00 weekday load to verify whether the evening surge represents people working late or automated architectural facade floodlights.


In [15]:
# ==============================================================================
# 8. EVENING CIRCUIT BREAKDOWN (17:00 - 22:00 WEEKDAYS)
# ==============================================================================
bldg_hourly = pd.read_csv(os.path.join(data_dir, "qn1_hourly_master_for_excel.csv"), parse_dates=['Date'], index_col='Date')
elec_cols = [c for c in bldg_hourly.columns if ('_Light(kW)' in c) or ('_Plug(kW)' in c)]

eve_mask = (bldg_hourly.index.dayofweek < 5) & (bldg_hourly['Is_Occupied'] == 0) & (bldg_hourly.index.hour >= 17) & (bldg_hourly.index.hour < 22)
eve_means = bldg_hourly.loc[eve_mask, elec_cols].mean()

f1_light_tot = eve_means[[c for c in elec_cols if 'Floor1' in c and '_Light' in c]].sum()
f1_plug_tot  = eve_means[[c for c in elec_cols if 'Floor1' in c and '_Plug' in c and 'z2' not in c]].sum()
server_tot   = eve_means['Floor1_z2_Plug(kW)']
up_light_tot = eve_means[[c for c in elec_cols if 'Floor1' not in c and '_Light' in c]].sum()
up_plug_tot  = eve_means[[c for c in elec_cols if 'Floor1' not in c and '_Plug' in c]].sum()
tot_eve      = f1_light_tot + f1_plug_tot + server_tot + up_light_tot + up_plug_tot

eve_breakdown_df = pd.DataFrame([
    {'End_Use_Category': 'Floor 1 Lighting (Facade & Atrium)', 'Mean_Power_kW': round(f1_light_tot, 2), 'Share_of_Evening_Load_%': round(f1_light_tot/tot_eve*100, 1)},
    {'End_Use_Category': 'Floor 1 Zone 2 Server (IT Data Center)', 'Mean_Power_kW': round(server_tot, 2), 'Share_of_Evening_Load_%': round(server_tot/tot_eve*100, 1)},
    {'End_Use_Category': 'Upper Floors (2–7) Office Lighting', 'Mean_Power_kW': round(up_light_tot, 2), 'Share_of_Evening_Load_%': round(up_light_tot/tot_eve*100, 1)},
    {'End_Use_Category': 'Upper Floors (2–7) Office Plugs (Desktops)', 'Mean_Power_kW': round(up_plug_tot, 2), 'Share_of_Evening_Load_%': round(up_plug_tot/tot_eve*100, 1)},
    {'End_Use_Category': 'Floor 1 Office Plugs', 'Mean_Power_kW': round(f1_plug_tot, 2), 'Share_of_Evening_Load_%': round(f1_plug_tot/tot_eve*100, 1)},
    {'End_Use_Category': 'Total Evening Active Power (17:00–22:00)', 'Mean_Power_kW': round(tot_eve, 2), 'Share_of_Evening_Load_%': 100.0}
])
display(eve_breakdown_df)


,End_Use_Category,Mean_Power_kW,Share_of_Evening_Load_%
0,Floor 1 Lighting (Facade & Atrium),94.15,57.7
1,Floor 1 Zone 2 Server (IT Data Center),15.39,9.4
2,Upper Floors (2–7) Office Lighting,41.93,25.7
3,Upper Floors (2–7) Office Plugs (Desktops),7.28,4.5
4,Floor 1 Office Plugs,4.37,2.7
5,Total Evening Active Power (17:00–22:00),163.12,100.0


> **Forensic Evening Audit Findings (17:00–22:00 Weekdays)**:
> * **Upper Floor Desktops Collapse**: Office plug receptacles across Floors 2 to 7 combined draw only **7.28 kW** (4.5% of load), indicating that computers are largely powered down and occupants have departed.
> * **Floor 1 Lighting Dominates**: Ground-floor facade and atrium lighting accounts for **57.7% (94.2 kW)** of the entire building's evening power draw.
> * **Upper-Floor Lighting Left On**: Office lighting on Floors 2 to 7 is the second-largest contributor at **25.7% (41.9 kW)** — this is the curfew-actionable component.
> * **Diagnosis**: The evening load peak is driven primarily by dusk architectural floodlights and upper-floor lights left on, not human overtime work (desktop plug loads have already collapsed).


## 9. Deliverable 01: The Waste Map (Per-Circuit, Per-Floor Spatial Analysis)

---

### Deliverable Alignment
> **01 Waste Map**: *A per-circuit, per-floor view of where after-hours baseload runs and how large it is.*

We visualize the spatial distribution of after-hours energy waste through a comprehensive **Heatmap Grid** (Floors 1 to 7 on the Y-axis vs. Zones/Circuits on the X-axis) alongside floor aggregate and end-use breakdowns.


In [16]:
# ==============================================================================
# 9. DELIVERABLE 01: SPATIAL WASTE MAP & BREAKDOWN PLOTS
# ==============================================================================
# 1. Create Per-Circuit, Per-Floor Waste Map Matrix (MWh)
waste_pivot = surplus_summary_df.pivot(index='Floor', columns='Zone_Col', values='Surplus_kWh_18M').fillna(0.0) / 1000.0
# Order floors from Floor 7 down to Floor 1 for architectural elevation perspective
floor_order = [f'Floor{i}' for i in range(7, 0, -1)]
waste_pivot = waste_pivot.reindex(floor_order)

# Reorder columns logically: z1_Light, z1_Plug, z2_Light, z2_Plug, etc.
sorted_cols = sorted(waste_pivot.columns, key=lambda x: (x.split('_')[0], 0 if 'Light' in x else 1))
waste_pivot = waste_pivot[sorted_cols]

# Plot Deliverable 01: Waste Map + Floor Totals + Category Breakdown
fig = plt.figure(figsize=(18, 10))

# Heatmap Matrix (Deliverable 01: Waste Map)
ax_map = plt.subplot2grid((2, 2), (0, 0), colspan=2)
sns.heatmap(waste_pivot, cmap='YlOrRd', annot=True, fmt='.1f', lw=0.5,
            cbar_kws={'label': '18-Month Addressable Energy Waste (MWh)'}, ax=ax_map)
ax_map.set_title("Deliverable 01: Chamchuri 5 Waste Map (Per-Circuit, Per-Floor Surplus in MWh)", fontsize=13, fontweight='bold')
ax_map.set_xlabel("Circuit Zone & Load Type", fontweight='bold')
ax_map.set_ylabel("Building Floor Elevation", fontweight='bold')

# Subplot 2: Floor Aggregate Totals
ax_floor = plt.subplot2grid((2, 2), (1, 0))
floor_s = pd.Series(floor_waste_totals).sort_values(ascending=False) / 1000.0
bars = ax_floor.bar(floor_s.index, floor_s.values, color='#F43F5E', width=0.55, edgecolor='black', lw=0.7)
ax_floor.set_ylabel("18-Month Surplus Energy (MWh)", fontweight='bold')
ax_floor.set_title("Addressable Energy Surplus by Floor (MWh)", fontweight='bold')
for b in bars:
    ax_floor.annotate(f"{b.get_height():.1f}M", (b.get_x() + b.get_width()/2., b.get_height()),
                      ha='center', va='bottom', fontsize=9, xytext=(0, 3), textcoords='offset points')
ax_floor.grid(True, alpha=0.3, axis='y')

# Subplot 3: End-Use Category Breakdown
ax_type = plt.subplot2grid((2, 2), (1, 1))
ax_type.pie([type_waste_totals['Lighting'], type_waste_totals['Plug Load']],
            labels=['Lighting (Corridors & Facade)', 'Plug Receptacles (Idle Desktops)'],
            autopct='%1.1f%%', colors=['#F59E0B', '#3B82F6'], startangle=140,
            explode=(0.04, 0), wedgeprops={'edgecolor': 'black', 'linewidth': 0.7})
ax_type.set_title("Addressable Surplus by End-Use Category", fontweight='bold')

plt.tight_layout()
plt.show()


/var/folders/1g/x_wv4b8n0zs5kkhz0j1mg7v40000gn/T/ipykernel_3494/4280376014.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


> **Deliverable 01: Waste Map Spatial Analysis**:
> * **Spatial Hotspots**: Upper floors (Floors 2 to 7) exhibit persistent idle desktop standby loads across individual zones.
> * **Standby vs. Waste Separation**: All 4 Critical Standby circuits register **0.0 MWh** of waste, while curfew-actionable office lighting and computer receptacles across the 61 preventable circuits are mapped to guide targeted curfew sweeps.


## 10. Addressable Energy & Utility Cost Quantification

### Thailand MEA Time-of-Use Tariff (Schedule 3.2 / 4.2: 12–24 kV Voltage Tier)
We evaluate avoided electricity expenditure using official **Metropolitan Electricity Authority (MEA)** rates for medium/large service connections at 12–24 kV (rates to be confirmed against MEA’s current 12–24 kV table):
* **Peak Energy Rate**: Monday to Friday, 09:00 – 22:00 @ **4.3297 THB / kWh**
* **Off-Peak Energy Rate**: Monday to Friday 22:00 – 09:00, all weekends, and official holidays @ **2.5849 THB / kWh**
* **7% VAT Avoidance**: Factored into net university utility cash savings.

*(Note: Hardware CapEx and payback periods are strictly excluded to maintain focus on operational reduction opportunities without speculative installation assumptions.)*


In [17]:
# ==============================================================================
# 10. MEA TARIFF 3.2 / 4.2 VALUATION & KPI BREAKDOWN (DYNAMIC BASELINE)
# ==============================================================================
PEAK_RATE_THB = 4.3297
OFFPEAK_RATE_THB = 2.5849
VAT_RATE = 0.07

# Build full excess dataframe across all 61 preventable circuits (4 critical standby circuits protected at 0.0 kWh)
all_unocc_excess = []
for circuit_id, excess_s in circuit_excess_18m.items():
    if circuit_id in critical_standby_circuits:
        continue
    all_unocc_excess.append(pd.DataFrame({
        'Circuit': circuit_id,
        'Floor': circuit_id.split('_')[0],
        'Type': 'Lighting' if '_Light' in circuit_id else 'Plug',
        'Excess_kWh': excess_s,
        'Hour': unocc_master.index.hour,
        'DayOfWeek': unocc_master.index.dayofweek,
        'Is_Holiday': pd.Series(unocc_master.index.date, index=unocc_master.index).isin(holiday_dates),
        'Is_Train_Period': (unocc_master['Is_Train_Period'] == 1)
    }, index=unocc_master.index))

waste_df = pd.concat(all_unocc_excess, axis=0)

# MEA Tariff: Peak is Mon-Fri 09:00 - 22:00 (non-holiday)
is_peak = (waste_df['DayOfWeek'] < 5) & (~waste_df['Is_Holiday']) & (waste_df['Hour'] >= 9) & (waste_df['Hour'] < 22)
waste_df['Tariff_Rate'] = np.where(is_peak, PEAK_RATE_THB, OFFPEAK_RATE_THB)
waste_df['Cost_THB'] = waste_df['Excess_kWh'] * waste_df['Tariff_Rate']
waste_df['Cost_w_VAT_THB'] = waste_df['Cost_THB'] * (1.0 + VAT_RATE)

# Clean 12-Month Headline Metrics (Jul 2018 - Jun 2019)
clean_12m_mask = waste_df['Is_Train_Period'] == 1
waste_12m = waste_df.loc[clean_12m_mask].copy()
annual_kwh_12m = waste_12m['Excess_kWh'].sum(skipna=True)
annual_cost_12m = waste_12m['Cost_THB'].sum(skipna=True)
annual_cost_vat_12m = annual_cost_12m * (1.0 + VAT_RATE)


# 1. Breakdown by Building Scope & End-Use (12M Baseline)
def assign_scope_group(row):
    if row['Floor'] == 'Floor1' and row['Type'] == 'Lighting':
        return 'Floor 1 (Zone 1) Entrance / Atrium Lighting'
    elif row['Floor'] == 'Floor1' and row['Type'] == 'Plug':
        return 'Floor 1 (Zones 1 & 3) General Plugs'
    elif row['Type'] == 'Lighting':
        return 'Upper Floors (2–7) Office Lighting'
    else:
        return 'Upper Floors (2–7) Office Desktop Plugs'

waste_12m['Scope_Group'] = np.where(
    (waste_12m['Floor'] == 'Floor1') & (waste_12m['Type'] == 'Lighting'),
    'Floor 1 (Zone 1) Entrance / Atrium Lighting',
    np.where(
        (waste_12m['Floor'] == 'Floor1') & (waste_12m['Type'] == 'Plug'),
        'Floor 1 (Zones 1 & 3) General Plugs',
        np.where(
            waste_12m['Type'] == 'Lighting',
            'Upper Floors (2–7) Office Lighting',
            'Upper Floors (2–7) Office Desktop Plugs'
        )
    )
)

scope_order = [
    ('Upper Floors (2–7) Office Lighting', '29 swept'),
    ('Floor 1 (Zone 1) Entrance / Atrium Lighting', '1 swept'),
    ('Floor 1 (Zones 1 & 3) General Plugs', '3 swept'),
    ('Upper Floors (2–7) Office Desktop Plugs', '28 swept'),
]

scope_rows = []
for grp_name, circ_cnt in scope_order:
    sub = waste_12m[waste_12m['Scope_Group'] == grp_name]
    kwh_v = sub['Excess_kWh'].sum(skipna=True)
    thb_v = sub['Cost_THB'].sum(skipna=True)
    scope_rows.append({
        'Building_Scope_and_End_Use': grp_name,
        'Circuits': circ_cnt,
        'Annual_Energy_Saved_kWh': round(kwh_v, 1),
        'Annual_Energy_Saved_MWh': round(kwh_v / 1000.0, 1),
        'Share_of_Savings_Pct': f"{kwh_v / annual_kwh_12m * 100.0:.1f}%",
        'Annual_Cost_Avoided_THB': round(thb_v, 2),
        'Annual_Cost_w_VAT_THB': round(thb_v * (1.0 + VAT_RATE), 2),
    })

scope_rows.append({
    'Building_Scope_and_End_Use': 'Critical Standby (IT Server Room + 3 Floor 1 Lights)',
    'Circuits': '4 protected',
    'Annual_Energy_Saved_kWh': 0.0,
    'Annual_Energy_Saved_MWh': 0.0,
    'Share_of_Savings_Pct': '0.0%',
    'Annual_Cost_Avoided_THB': 0.0,
    'Annual_Cost_w_VAT_THB': 0.0,
})
scope_rows.append({
    'Building_Scope_and_End_Use': 'Total 12M Dynamic Baseline Impact',
    'Circuits': '65 total (61 swept)',
    'Annual_Energy_Saved_kWh': round(annual_kwh_12m, 1),
    'Annual_Energy_Saved_MWh': round(annual_kwh_12m / 1000.0, 1),
    'Share_of_Savings_Pct': '100.0%',
    'Annual_Cost_Avoided_THB': round(annual_cost_12m, 2),
    'Annual_Cost_w_VAT_THB': round(annual_cost_vat_12m, 2),
})

scope_kpi_df = pd.DataFrame(scope_rows)
print("=== KPI BREAKDOWN BY BUILDING SCOPE & END-USE (12M DYNAMIC BASELINE) ===")
display(scope_kpi_df)

# 2. Breakdown by Floor (Floors 1 to 7)
floor_rows = []
for f_idx in range(1, 8):
    f_key = f'Floor{f_idx}'
    sub_f = waste_12m[waste_12m['Floor'] == f_key]
    kwh_f = sub_f['Excess_kWh'].sum(skipna=True)
    thb_f = sub_f['Cost_THB'].sum(skipna=True)
    floor_rows.append({
        'Floor': f'Floor {f_idx}',
        'Annual_Energy_Saved_kWh': round(kwh_f, 1),
        'Annual_Energy_Saved_MWh': round(kwh_f / 1000.0, 1),
        'Share_of_Savings_Pct': f"{kwh_f / annual_kwh_12m * 100.0:.1f}%",
        'Annual_Cost_Avoided_THB': round(thb_f, 2),
        'Annual_Cost_w_VAT_THB': round(thb_f * (1.0 + VAT_RATE), 2),
    })
floor_kpi_df = pd.DataFrame(floor_rows)
print("\n=== KPI BREAKDOWN BY FLOOR (FLOORS 1 TO 7) ===")
display(floor_kpi_df)

# ==============================================================================
# EXPORT CIRCUIT CLASSIFICATION & BASELOAD AUDIT SUMMARY (DIGITAL TWIN SYNC)
# ==============================================================================
master_sync_df = pd.merge(
    feat_df[['Circuit_ID', 'Floor', 'Zone_Col', 'Type', 'Baseload_kW', 'Unocc_to_Occ_Ratio', 'OffHours_CV', 'Classification', 'Behaviour_Group']],
    surplus_summary_df[['Circuit_ID', 'Unocc_Mean_kW', 'Surplus_kWh_18M', 'Annual_Surplus_kWh']],
    on='Circuit_ID',
    how='left'
)
avg_thb_per_kwh = annual_cost_12m / annual_kwh_12m
master_sync_df['Annual_Avoided_Cost_THB'] = (master_sync_df['Annual_Surplus_kWh'] * avg_thb_per_kwh).round(2)

export_path = os.path.join(data_dir, 'qn1_circuit_waste_and_classification.csv')
master_sync_df.to_csv(export_path, index=False)
print(f"[OK] Master Circuit Classification exported -> {os.path.basename(export_path)}")

# Overall after-hours reduction percentages
all_circ_cols = list(circuit_excess_18m.keys())
unocc_12m = unocc_master[unocc_master['Is_Train_Period'] == 1]
total_unocc_kwh_12m = unocc_12m[all_circ_cols].sum().sum()
upper_cols = [c for c in all_circ_cols if not c.startswith('Floor1_')]
upper_unocc_kwh_12m = unocc_12m[upper_cols].sum().sum()
upper_saved_kwh_12m = waste_12m[waste_12m['Floor'] != 'Floor1']['Excess_kWh'].sum(skipna=True)

audit_summary_df = pd.DataFrame([
    {'Metric': 'Annual Preventable Energy Saved (12M Dynamic Baseline)', 'Value': f'{annual_kwh_12m:,.1f} kWh / year ({annual_kwh_12m/1000.0:.1f} MWh / year)'},
    {'Metric': 'Annual Utility Cost Avoided (MEA Schedule 4.2 TOU excl. VAT)', 'Value': f'THB {annual_cost_12m:,.2f} / year'},
    {'Metric': 'Total Utility Cost Avoided (Including 7% VAT)', 'Value': f'THB {annual_cost_vat_12m:,.2f} / year'},
    {'Metric': 'Building-Wide After-Hours Energy Reduction', 'Value': f'-{annual_kwh_12m / total_unocc_kwh_12m * 100.0:.1f}% (of {total_unocc_kwh_12m/1000.0:.1f} MWh unoccupied total)'},
    {'Metric': 'Upper Office Floors (2–7) After-Hours Energy Reduction', 'Value': f'-{upper_saved_kwh_12m / upper_unocc_kwh_12m * 100.0:.1f}% (of {upper_unocc_kwh_12m/1000.0:.1f} MWh upper-floor unoccupied total)'}
])
display(audit_summary_df)


=== KPI BREAKDOWN BY BUILDING SCOPE & END-USE (12M DYNAMIC BASELINE) ===


,Building_Scope_and_End_Use,Circuits,Annual_Energy_Saved_kWh,Annual_Energy_Saved_MWh,Share_of_Savings_Pct,Annual_Cost_Avoided_THB,Annual_Cost_w_VAT_THB
0,Upper Floors (2–7) Office Lighting,29 swept,86123.5,86.1,49.0%,297890.27,318742.59
1,Floor 1 (Zone 1) Entrance / Atrium Lighting,1 swept,50143.4,50.1,28.5%,153051.26,163764.85
2,Floor 1 (Zones 1 & 3) General Plugs,3 swept,22647.7,22.6,12.9%,67314.18,72026.17
3,Upper Floors (2–7) Office Desktop Plugs,28 swept,16739.0,16.7,9.5%,52358.81,56023.92
4,Critical Standby (IT Server Room + 3 Floor 1 L...,4 protected,0.0,0.0,0.0%,0.00,0.00
5,Total 12M Dynamic Baseline Impact,65 total (61 swept),175653.6,175.7,100.0%,570614.52,610557.53



=== KPI BREAKDOWN BY FLOOR (FLOORS 1 TO 7) ===


,Floor,Annual_Energy_Saved_kWh,Annual_Energy_Saved_MWh,Share_of_Savings_Pct,Annual_Cost_Avoided_THB,Annual_Cost_w_VAT_THB
0,Floor 1,72791.1,72.8,41.4%,220365.44,235791.02
1,Floor 2,10992.1,11.0,6.3%,37780.99,40425.66
2,Floor 3,14848.1,14.8,8.5%,52300.94,55962.00
3,Floor 4,16575.3,16.6,9.4%,55409.25,59287.90
4,Floor 5,18242.1,18.2,10.4%,64428.43,68938.42
5,Floor 6,22143.1,22.1,12.6%,74015.85,79196.96
6,Floor 7,20061.7,20.1,11.4%,66313.61,70955.56


[OK] Master Circuit Classification exported -> qn1_circuit_waste_and_classification.csv


,Metric,Value
0,Annual Preventable Energy Saved (12M Dynamic B...,"175,653.6 kWh / year (175.7 MWh / year)"
1,Annual Utility Cost Avoided (MEA Schedule 4.2 ...,"THB 570,614.52 / year"
2,Total Utility Cost Avoided (Including 7% VAT),"THB 610,557.53 / year"
3,Building-Wide After-Hours Energy Reduction,-26.1% (of 673.4 MWh unoccupied total)
4,Upper Office Floors (2–7) After-Hours Energy R...,-77.8% (of 132.1 MWh upper-floor unoccupied to...


> **Energy & Utility Cost Avoidance Findings (Dynamic Baseline)**:
> * **Annual Preventable Energy Saved**: **175,653.6 kWh / year** (**175.7 MWh / year**) across the 61 curfew-eligible circuits under the 12-month dynamic baseline (`4 Critical Standby` circuits protected at `0.0 kWh`).
> * **Annual Utility Cost Avoided**: **฿570,614.52 THB / year** excl. VAT (**฿610,557.53 THB / year** incl. 7% VAT) under MEA Schedule 4.2 Time-of-Use rates.
> * **Relative After-Hours Reduction**: Eliminating preventable surplus reduces total building after-hours lighting and plug consumption by **−26.1%** (`175.7 / 673.4 MWh`), and cuts **Upper Office Floors (Floors 2–7)** after-hours consumption by **−77.8%** (`102.9 / 132.1 MWh`).
> * **Savings Breakdown by Scope & End-Use**:
>   * **Upper Floors (2–7) Office Lighting (`29 circuits`)**: **86.1 MWh/yr** (`49.0%` of savings, **฿318,743/yr** incl. VAT).
>   * **Floor 1 (Zone 1) Entrance / Atrium Lighting (`1 circuit`)**: **50.1 MWh/yr** (`28.5%` of savings, **฿163,765/yr** incl. VAT) — observed at `0.00 kW` during the Sunday calibration window on 9 of 52 training weeks, confirming overnight operation is a timer oversight rather than an essential safety load.
>   * **Floor 1 (Zones 1 & 3) General Plugs (`3 circuits`)**: **22.6 MWh/yr** (`12.9%` of savings, **฿72,026/yr** incl. VAT).
>   * **Upper Floors (2–7) Office Desktop Plugs (`28 circuits`)**: **16.7 MWh/yr** (`9.5%` of savings, **฿56,024/yr** incl. VAT).
>   * **Critical Standby (`4 circuits`: `Floor1_z2_Plug` Server Room + `Floor1_z2, z3, z4_Light`)**: **0.0 MWh/yr** (**100% Protected**).

## 11. Model Generalization: Out-of-Sample Holdout Evaluation (July – Dec 2019)

To evaluate whether calibrated baselines ($P_{2.5}$) and unsupervised classifications generalize without seasonal confounding, we conduct a like-for-like comparison between identical calendar periods: **Jul–Dec 2018 (Train Window)** vs. **Jul–Dec 2019 (Held-Out Test Window)**.

### Floor 6 Telemetry Truncation Disclosure
* **Floors 1, 2, 3, 4, 5, and 7** contain the full 264,960 minutes (184 days) across both periods.
* **Floor 6 telemetry terminates on October 22, 2019**, providing only 163,528 minutes (missing 101,432 minutes / ~70.4 days in the test window).
* To prevent missing data from distorting the generalizability audit, we report both:
  1. **Building-Wide (All 7 Floors)**: Demonstrates empirical variance (−5.03%) under raw telemetry.
  2. **Complete Floors (Floors 1–5, 7)**: Directly evaluates identical 6-month windows across all 100% complete floors (−1.75% variance), confirming high baseline stability across years.


In [18]:
# ==============================================================================
# 11. OUT-OF-SAMPLE TEST EVALUATION (JULY - DEC 2018 VS. JULY - DEC 2019)
# ==============================================================================
jd18_mask = (waste_df.index >= '2018-07-01 00:00:00') & (waste_df.index <= '2018-12-31 23:59:59')
jd19_mask = (waste_df.index >= '2019-07-01 00:00:00') & (waste_df.index <= '2019-12-31 23:59:59')

# 1. Building-wide comparison (All 7 Floors)
jd18_kwh = waste_df.loc[jd18_mask, 'Excess_kWh'].sum(skipna=True)
jd19_kwh = waste_df.loc[jd19_mask, 'Excess_kWh'].sum(skipna=True)
var_all_pct = (jd19_kwh - jd18_kwh) / jd18_kwh * 100.0

# 2. Complete Floors comparison (Excluding Floor 6 due to 70-day truncation in late 2019)
nof6_mask = ~waste_df['Circuit'].str.startswith('Floor6_')
jd18_kwh_nof6 = waste_df.loc[jd18_mask & nof6_mask, 'Excess_kWh'].sum(skipna=True)
jd19_kwh_nof6 = waste_df.loc[jd19_mask & nof6_mask, 'Excess_kWh'].sum(skipna=True)
var_nof6_pct = (jd19_kwh_nof6 - jd18_kwh_nof6) / jd18_kwh_nof6 * 100.0

holdout_audit_df = pd.DataFrame([
    {
        'Evaluation_Scope': 'All 7 Floors (Raw Telemetry)',
        'Jul_Dec_2018_Train_kWh': round(jd18_kwh, 1),
        'Jul_Dec_2019_Test_kWh': round(jd19_kwh, 1),
        'Variance_%': f"{var_all_pct:+.2f}%",
        'Data_Integrity_Notes': 'Floor 6 missing ~70 days (101,432 mins) in Test window'
    },
    {
        'Evaluation_Scope': 'Complete Floors (Floors 1-5, 7)',
        'Jul_Dec_2018_Train_kWh': round(jd18_kwh_nof6, 1),
        'Jul_Dec_2019_Test_kWh': round(jd19_kwh_nof6, 1),
        'Variance_%': f"{var_nof6_pct:+.2f}%",
        'Data_Integrity_Notes': '100% complete records across both identical 6-month horizons'
    }
])

display(holdout_audit_df)
print(f"[VERIFIED] Out-of-sample stability confirmed: {var_nof6_pct:+.2f}% variance across identical seasonal periods on complete floors.")
print(f"  Building-wide variance including truncated Floor 6: {var_all_pct:+.2f}%.")


,Evaluation_Scope,Jul_Dec_2018_Train_kWh,Jul_Dec_2019_Test_kWh,Variance_%,Data_Integrity_Notes
0,All 7 Floors (Raw Telemetry),95459.8,90657.5,-5.03%,"Floor 6 missing ~70 days (101,432 mins) in Tes..."
1,"Complete Floors (Floors 1-5, 7)",84375.5,82895.4,-1.75%,100% complete records across both identical 6-...


[VERIFIED] Out-of-sample stability confirmed: -1.75% variance across identical seasonal periods on complete floors.
  Building-wide variance including truncated Floor 6: -5.03%.


> **Model Generalization Finding (Zero Data Leakage Audit)**:
> * **Like-for-Like Seasonal Comparison**: On the 6 complete floors (1–5, 7), preventable surplus in the unobserved holdout window (Jul–Dec 2019: **82,895 kWh**) is within **−1.75%** of the identical calendar window in the training year (Jul–Dec 2018: **84,376 kWh**).
> * **Building-Wide Variance (−5.03%)**: Including Floor 6 (90,658 kWh vs. 95,460 kWh), the larger gap is attributable to Floor 6 telemetry terminating on 22 Oct 2019 (~70 missing days), not to model drift.
> * **Robust Operational Generalization**: Calibrating the Time-of-Week $P_{2.5}(d, h)$ matrix strictly on training data yields exceptionally consistent, generalizable out-of-sample predictions.


In [19]:
# ===== 12. BASELOAD VALIDATION: OUT-OF-SAMPLE COVERAGE CHECK =====
cols  = [c for c in all_circuit_cols if c not in critical_standby_circuits]
unocc = master_hourly[master_hourly['Is_Unoccupied'] == 1]
train = unocc[unocc['Is_Train_Period'] == 1]
test  = unocc[unocc.index >= '2019-07-01']
dow_h = lambda df: pd.MultiIndex.from_arrays([df.index.dayofweek, df.index.hour])

def coverage(df, matrix):
    base = matrix.reindex(dow_h(df))[cols].to_numpy()
    act  = df[cols].to_numpy()
    ok = ~np.isnan(act) & ~np.isnan(base)
    return (act >= base)[ok].mean()

cov_df = pd.DataFrame([
    {'Split': 'Train (Jul 2018 – Jun 2019)',   'Target_Coverage': 0.975, 'Observed_Coverage': round(coverage(train, dynamic_base_matrix), 3)},
    {'Split': 'Test (Jul – Dec 2019, Unseen)', 'Target_Coverage': 0.975, 'Observed_Coverage': round(coverage(test, dynamic_base_matrix), 3)},
])
display(cov_df)

,Split,Target_Coverage,Observed_Coverage
0,Train (Jul 2018 – Jun 2019),0.975,0.974
1,"Test (Jul – Dec 2019, Unseen)",0.975,0.967


**Baseload validation.** Baseload is estimated as the 2.5th percentile of after-hours load per `(day, hour)` slot, so it should sit below actual load about 97.5% of the time. Coverage was **97.4%** on the training year and **96.7%** on the unseen Jul–Dec 2019 window.

---

### Executive Summary & Deliverables Alignment

| Project Dimension | Analytical Finding / Deliverable | Strategic Management Implication |
| :--- | :--- | :--- |
| **Operating Boundary** | **Mon–Fri 08:00–17:00** (Stable across years) | Clear physical demarcation between occupied surge (+85 kW) and unoccupied curfew hours. |
| **Reference Calibration Window** | **Sundays 02:00–04:00 AM** | Lowest-occupancy, most stable overnight window (~79.0 kW), capturing steady-state baseload. |
| **Baseline Architecture** | **Dynamic Time-of-Week $P_{2.5}(d, h)$** | Dynamic 168-hour baseline accommodates natural departure gradients while preserving quiescent floors. |
| **Deliverable 01: Waste Map** | **Generated Heatmap (Floors 1–7)** | Floor 1 facade/atrium lighting drives ~58% of evening load; upper-floor lighting left on adds ~26%. |
| **Deliverable 02: Classification** | **4 Critical Standby / 61 Preventable Waste** | Data-driven K-Means ($k=2$, Silhouette 0.735) isolates server & infrastructure load from 61 office loads. |
| **Annual Preventable Energy** | **175,653.6 kWh / year** | Clean 12-month dynamic baseline (accommodating evening departure shoulder). |
| **Annual Utility Cost Avoidance** | **฿570,614.52 THB / year** (excl. VAT)<br>**฿610,557.53 THB / year** (incl. 7% VAT) | Direct cash savings potential via automated BEMS curfew sweeps (MEA Schedule 4.2 TOU; rates to be confirmed against MEA’s current 12–24 kV table). |
| **Model Generalization (Test Set)** | **−1.75% Variance (complete floors)**<br>−5.03% incl. truncated Floor 6 | Like-for-like Jul–Dec 2019 holdout vs. Jul–Dec 2018 confirms highly stable, generalizable baselines. |
| **Baseload Coverage Check** | **97.4% (Train) / 96.7% (Test)** vs. 97.5% target | Actual after-hours load sits above the $P_{2.5}(d, h)$ floor at the expected quantile rate on unseen data. |
| **Digital Twin Transfer** | **`qn1_dynamic_baseload_matrix.csv`** | Direct synchronization with building 3D model and savings simulator webapp. |
